In [3]:
import dataclasses
from dataclasses import dataclass, fields
from typing import TypeVar, Type

data1 = {
	"name": "Jack",
	"age": 21,
	"address": {"city": "Jefferson", "zipcode": 94118}
}

@dataclass
class Address:
	city: str
	zipcode: int

@dataclass
class Person: 
	name: str
	age: int
	address: Address

person2 = Person(**data1)
print(person2) # problematic because it doesn't create the class but just the dict

T = TypeVar('T') # dynamic placeholder for a class

def build_dataclass(datacls: Type[T], d: dict) -> T:
	
	# validate no keys are wrong
	valid_fields = {f.name for f in fields(datacls)}
	if invalid_keys := d.keys() - valid_fields: 
		raise ValueError(f"Invalid keys: {invalid_keys}")
	
	kwargs = {}
	for f in fields(datacls): # iterate over classes fields
		if f.name not in d: 
			continue # next field in the dataclass - this can be okay because of defaults
		value = d[f.name] # value in the dict
		
		# now I'm concerned if f is a nested class, I need to verify the dict follows this class' results
		if isinstance(value, dict) and dataclasses.is_dataclass(f.type):
			kwargs[f.name] = build_dataclass(f.type, value)
		else: 
			kwargs[f.name] = value
	return datacls(**kwargs)

person1 = build_dataclass(Person, data1)
print(person1)

Person(name='Jack', age=21, address={'city': 'Jefferson', 'zipcode': 94118})
Person(name='Jack', age=21, address=Address(city='Jefferson', zipcode=94118))


In [29]:
import torch
from train.utils import build_batch
import uuid
from train.store import Trajectory
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

prompt = "What is the tallest tower in the world?"
response = "Burj Khalifa"
prompt = tokenizer(prompt, return_tensors="pt")
response = tokenizer(response, return_tensors="pt")
print(prompt)
print(len(response['input_ids'][0].numpy()))
rollout_logprobs = torch.randn(len(response['input_ids'][0]), dtype=torch.float32)

trajectory = Trajectory(
	task_id=uuid.uuid4(),
	prompt_token_ids=prompt,
	response_token_ids=response['input_ids'][0],
	rollout_logprobs=rollout_logprobs, 
	policy_version=1,
	hint="Its in Abu Dahbi",
	judge_score=0.3,
	# step_spans=
	# loss_mask=
)

{'input_ids': tensor([[  101,  2054,  2003,  1996, 13747,  3578,  1999,  1996,  2088,  1029,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
6


Torch.gather

- index along the dim you want to pull from
- index has to be along the same dimension as you're pulling from, so basically shapes have to match


In [44]:
range = torch.arange(10)
range = range.unsqueeze(0).expand(10,-1)
print(range.shape)

index=torch.tensor([0,3,4,5,6,7,1,3,4,9]).unsqueeze(0)

torch.gather(range, dim=-1,index=index)

torch.Size([10, 10])


tensor([[0, 3, 4, 5, 6, 7, 1, 3, 4, 9]])

In [10]:
from pydantic import BaseModel, Field

class Address(BaseModel):
	city: str
	zipcode: int

class Person(BaseModel): 
	name: str
	age: int
	address: Address

data1 = {
	"name": "Jack",
	"age": 21,
	"address": {"city": "Jefferson", "zipcode": 94118} # auto-applies class
}

person = Person(**data1)
print(person)

name='Jack' age=21 address=Address(city='Jefferson', zipcode=94118)


## Model training step-by-step

#### No TIR

Initialization

1. Initialization
   - compile vLLM wheels
   - compile trainer - sharded model using FSDP2 (we've selectively sharded each layer using fully_shard(), giving us per-layer control), basically implementing ZeRO3
   - setup weight sync

2. vLLM generation
   - save full log-probs, prompt_token_ids, response_token_ids, text, task_id
   - generate the hint (manage concurrency using a semaphore), grade the response (also semaphore for grading)
     - `grader` class
   - return the `RolloutResult` object

3. Store it
   - store in a `Trajectory` object that includes the hint
   - `build_batch()` - compute the attention_mask (for forward pass by teacher + student model later on), response_mask (mask used for training, masks the prompt + pad + tool call tokens)
     - enables us to do the forward pass with the student + teacher (off-policy)
   - `gather_response_logprobs()` - builds the training_batch() by returning just the logits we want to train on

4. Rollout loop
   - define a stop Event() object

5. Training
   - 4 trainer GPUs - each processing a minibatch, with gradient accumulation only on the last minibatch of the 'microbatch' (minibatch=16 per GPU at a time, microbatch=64 each loss.backward(), batch_size=256 microbatch $\times$ grad_accum)
   - Weight are split in 1/6 across all 4 GPUs
   - Weights are reduce-scattered across GPUs in the forward pass (4 GPUs running 4 different batches with cross-communication of weights)
   - Gradients are reduce-scattered, activations are stored in the computational graph so backprop is possible on each GPU's slice of the weights

6. Weight sync
	- weight sync 
